# Stage 09B — Co-VAE representative paper-grid full-training reference

Attach the reviewed runner and the unchanged CoVAE source with `data/davis/`. Select a Kaggle T4 session. Run these four code cells in order. This notebook does not install packages or start a grid, later stage, or DCGAN-DTA job. Cell 1 checks inputs/environment without model construction or training.

In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

EXPECTED_RUNNER_SHA256 = "c5be744f18ed0cf6298ecbc32c50380362122adf8c34988745b7ad023b31c7b2"
matches = list(Path("/kaggle/input").rglob("stage09b_covae_full_kaggle.py"))
if len(matches) != 1:
    raise RuntimeError(f"Attach exactly one Stage 09B runner; found {matches}")
RUNNER = matches[0]
OUT = Path("/kaggle/working/stage09b_full")
CONSOLE = Path("/kaggle/working/stage09b_full_console.log")
def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()
assert file_sha(RUNNER) == EXPECTED_RUNNER_SHA256, "Uploaded runner differs from reviewed runner"
RUN_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONHASHSEED="0",
               PYTHONDONTWRITEBYTECODE="1",
               STAGE09B_EXPECTED_RUNNER_SHA256=EXPECTED_RUNNER_SHA256)
print("Runner:", RUNNER)
checked = subprocess.run([sys.executable, "-B", "-u", str(RUNNER), "--preflight"],
                         env=RUN_ENV, capture_output=True, text=True)
print(checked.stdout)
if checked.stderr:
    print(checked.stderr)
if checked.returncode:
    raise RuntimeError(f"Environment/input preflight failed: exit {checked.returncode}")
PREFLIGHT = json.loads(checked.stdout)
print("Co-VAE source:", PREFLIGHT["source"])
print("Preflight passed. No model or training was executed.")


## Cell 2 — launch one supervised run

Background execution keeps Cell 3 available for live observation. The runner has an exclusive output lock, preserves nonempty output directories, and has no wall-time kill. Its RSS/GPU monitor may stop an unsafe run. Interrupting Cell 3 does not stop the worker. Do not start a second run.

In [ ]:
# One background supervisor lets Cell 3 observe training in this notebook.
# The runner supervises its own single worker; monitoring cells only read files.
if "stage09b_job" in globals() and stage09b_job.poll() is None:
    raise RuntimeError("The previous Stage 09B job is still running")
if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError("Preserve the nonempty output directory; no automatic retry")
assert file_sha(RUNNER) == EXPECTED_RUNNER_SHA256
with CONSOLE.open("x", encoding="utf-8") as console_stream:
    stage09b_job = subprocess.Popen(
        [sys.executable, "-B", "-u", str(RUNNER)], env=RUN_ENV,
        cwd="/kaggle/working", stdout=console_stream,
        stderr=subprocess.STDOUT, start_new_session=True)
print("Supervisor PID:", stage09b_job.pid)
print("Output:", OUT)
print("Supervisor console:", CONSOLE)


## Cell 3 — read-only monitoring

Refreshes every five seconds until final artifacts exist. No held-out predictions or metrics are read during training. Training loss components are batch-size-weighted means; branch losses combine reconstruction and KL. CUDA peaks are cumulative since construction. Physical GPU samples may include other processes.

In [ ]:
import csv, time
from collections import deque
from IPython.display import clear_output

while True:
    clear_output(wait=True)
    live_path = OUT / "live_status.json"
    try:
        live = json.loads(live_path.read_text())
    except (OSError, json.JSONDecodeError):
        live = {}
    epoch = live.get("epoch", live.get("completed_epochs", 0))
    elapsed = live.get("elapsed_seconds", live.get("total_parent_seconds"))
    print(f"Phase: {live.get('phase', 'starting')}; current epoch {epoch}/100; completed {live.get('completed_epochs', 0)}")
    print("Elapsed seconds:", elapsed)
    print("Current batch:", live.get("batch"), "/", live.get("batches_per_epoch"))
    print("Latest completed epoch:", live.get("latest_epoch_metrics", live.get("last_epoch")))
    print("Latest batch metrics:", live.get("latest_batch_metrics"))
    cuda = live.get("cuda_memory", live)
    print("CUDA current allocated/reserved MiB:",
          [round(cuda.get(k, 0) / 1024**2, 2)
           for k in ("cuda_allocated_bytes", "cuda_reserved_bytes")])
    print("CUDA peak allocated/reserved MiB:",
          [round(cuda.get(k, 0) / 1024**2, 2)
           for k in ("cuda_peak_allocated_bytes", "cuda_peak_reserved_bytes")])
    warnings = list(live.get("warnings", []))
    latest_resource = {}
    try:
        with (OUT / "resource_history.csv").open(newline="") as stream:
            latest_resource = next(iter(deque(csv.DictReader(stream), maxlen=1)), {})
        print("Latest physical GPU and host RSS sample:", latest_resource)
        rss = float(latest_resource.get("worker_rss_bytes") or 0)
        gpu = float(latest_resource.get("gpu_used_mib") or 0)
        gpu_limit = PREFLIGHT["physical_gpu"]["safety_limit_mib"]
        if rss > 0.8 * 12 * 1024**3:
            warnings.append("Host RSS above 80% of safety limit")
        if gpu > 0.8 * gpu_limit:
            warnings.append("Physical GPU memory above 80% of safety limit")
    except (OSError, ValueError):
        warnings.append("Resource CSV is starting or currently incomplete")
    if live.get("timestamp") and time.time() - live["timestamp"] > 120:
        warnings.append("Worker status has not refreshed for 120 seconds; inspect console")
    print("Warnings:", warnings or "none")
    print("This cell does not stop, retry, reseed, evaluate, or modify training.")
    if (OUT / "final_summary.json").exists() and (OUT / "artifact_hashes.json").exists():
        print("Final artifacts ready. Run Cell 4.")
        break
    if "stage09b_job" in globals() and stage09b_job.poll() is not None:
        print("Supervisor exited:", stage09b_job.returncode, "; run Cell 4 or inspect console")
        break
    time.sleep(5)


## Cell 4 — final evidence and integrity

Run after Cell 3 reports completion or failure. Keep all artifacts for repository review. A completed job is a representative paper-grid reference, not proof of exact paper reproduction. No automatic retry or cleanup occurs.

In [ ]:
if "stage09b_job" in globals() and stage09b_job.poll() is None:
    raise RuntimeError("Training/finalization is still running; continue Cell 3")
summary_path = OUT / "final_summary.json"
if not summary_path.exists():
    if CONSOLE.exists():
        print("\n".join(CONSOLE.read_text(errors="replace").splitlines()[-40:]))
    raise RuntimeError("Final summary missing. Do not rerun automatically.")
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
print("\nNative held-out metrics:", summary.get("native_test_metrics"))
print("External saved-vector checks:", summary.get("external_checks"))
print("\nArtifacts:")
for path in sorted(OUT.iterdir()):
    if path.is_file():
        print(path.name, path.stat().st_size, "bytes")
hashes = json.loads((OUT / "artifact_hashes.json").read_text())
print("\nArtifact hashes:", json.dumps(hashes, indent=2))
mismatched = [name for name, digest in hashes["files"].items()
              if not (OUT / name).is_file() or file_sha(OUT / name) != digest]
print("\nSupervisor console tail:")
if CONSOLE.exists():
    print("\n".join(CONSOLE.read_text(errors="replace").splitlines()[-30:]))
print("\nNative training/evaluation log tail:")
log = OUT / "training.log"
if log.exists():
    print("\n".join(log.read_text(errors="replace").splitlines()[-25:]))
print("\nStatus:", summary["status"], "; reason:", summary.get("stopping_reason"))
if mismatched:
    raise RuntimeError(f"Artifact integrity failed: {mismatched}")
if summary["status"] != "completed":
    raise RuntimeError("Stage 09B execution failed/invalid; retain all evidence")
if "stage09b_job" in globals():
    assert stage09b_job.returncode == 0
print("Execution completed; repository forensic review is still needed before closing Stage 09B.")
